In [0]:
%sql
LIST '/Volumes/marathos/default/raw/marathon/'

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"
spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
spark.sql(f"LIST '{VOLUME_PATH}/marathon'").display()

In [0]:
%sql
SHOW SCHEMAS IN marathos;

In [0]:
BASE_DIR = "/Volumes/marathos/default/raw/"

schema = (
    spark.read.format("csv")
    .options(header=True, inferschema=True)
    .load(f"{BASE_DIR}/marathon/TWO_CENTURIES_OF_UM_RACES.csv")
    .schema
)

schema

In [0]:
%sql
FROM marathos.bronze.raw_marathos LIMIT 5;

# EDA — Bronze layer source data

In [0]:
from pyspark.sql.functions import col, count, when, countDistinct, desc, avg

RAW_PATH = "/Volumes/marathos/default/raw/marathon/"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv(RAW_PATH))

print(f"Rows:    {df.count():,}")
print(f"Columns: {len(df.columns)}")

In [0]:
# schema
df.printSchema()

In [0]:
# descriptive summary of numerical fields
df.select(
    "Year of event",
    "Event number of finishers",
    "Athlete year of birth",
    "Athlete ID"
).describe().display()

In [0]:
# null counts per column
nulls = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])
nulls.display()

In [0]:
# number of unique events
df.select(countDistinct("Event name").alias("unique_events")).display()

In [0]:
# Age distribution with Plotly
from pyspark.sql.functions import col
import plotly.express as px

ages = (df
    .withColumn("age", col("Year of event") - col("Athlete year of birth"))
    .filter(col("age").between(15, 100))
    .groupBy("age").count()
    .orderBy("age")
    .toPandas())

fig = px.bar(ages, x="age", y="count", title="Age distribution of runners")
fig.show()

In [0]:
 #most represented countries
 df.groupBy("Athlete country") \
  .count() \
  .orderBy(desc("count")) \
  .limit(20) \
  .display()

In [0]:
# Unique values of `Event distance/length`
df.select("Event distance/length").distinct().display()

In [0]:
# Counting event units
from pyspark.sql.functions import col, lower, regexp_replace, desc

units = (df
    .withColumn("unit", lower(regexp_replace(col("Event distance/length"), r"[0-9.]", "")))
    .groupBy("unit").count()
    .orderBy(desc("count")))
units.display()